# Classification of Wikipedia Articles
Wikipedia is an encyclopedia that covers a large amount of diverse topics. All articles are created, corrected and updated by individuals. The goal is to correctly document as many topics as possible by collecting the knowledge of a large number of people. However, some articles stand out due to their completeness, scope and presentation, and for this they are marked with the distinction of the Excellent Article. 

As part of the Natural Language Processing lecture, a classification of Wikipedia articles is to be carried out as a sub-task of an assignment with the goal of being able to identify excellent articles. This notebook contains the code to accomplish this goal and is structured as follows:

1. [Imports](#1-imports)
2. [Load Preprocessed Data](#2-load-preprocessed-data)
3. [Prepare Dataset for Neural Network](#3-prepare-data-for-neural-network) <br>
	3.1 [Tokenize Words](#31-tokenize-words) <br>
	3.2 [Clip Text Length](#32-clip-text-length) <br>
4. [Split Dataset](#4-split-dataset)
5. [Train Neural Network](#5-train-neural-network) <br>
	5.1 [Define Neural Network Model]() <br>
	5.2 [Compile Neural Network Model]() <br>
	5.3 [Train the Neural Network]() <br>
6. [Validation of Results]()
7. [Conclusion]()


## 1. Imports
Import the requiered libraties into the notebook.
If some libraries are not installed, you can use the `requierements.txt` and run
```
$ pip install -r requirements.txt
```
in the terminal.

In [1]:
# Import data science library
import pandas as pd
import numpy as np

# Import Pre-Processing libraries
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Import neural network framework & layers
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Conv1D, LSTM, Dense
from tensorflow.keras.layers import BatchNormalization, Dropout, MaxPooling1D
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.regularizers import L1L2

# Import classification metrics
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from imblearn.metrics import geometric_mean_score

# Import visualization libraries
import plotly.graph_objects as go
from prettytable import PrettyTable

## 2. Load Preprocessed Data

In [2]:
dataframe = pd.read_pickle("../../Data/processed_dataset.pkl")

In [3]:
X = np.array(dataframe["text"].values)
y = np.asanyarray(dataframe["label"].values).astype(np.int16)

In [4]:
np.unique(y, return_counts=True)

(array([0, 1], dtype=int16), array([4193, 2794]))

## 3. Prepare Data for Neural Network

### 3.1 Tokenize Words

In [5]:
tokenizer = Tokenizer(
    num_words=10000,
    filters='!"„“#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)
tokenizer.fit_on_texts(X)

### 3.2 Clip Text Length

In [6]:
X_token = pad_sequences(tokenizer.texts_to_sequences(X), maxlen=10000)

## 4. Split Dataset

In [7]:
X_train, X_rest, y_train, y_rest = train_test_split(
    X_token, 
    y,
    stratify=y, 
    test_size=0.3,
    random_state=456
)

X_test, X_val, y_test, y_val = train_test_split(
    X_rest,
    y_rest,
    stratify=y_rest,
    test_size=0.5,
    random_state=456
)

In [8]:
print(len(X_train), len(X_test), len(X_val))

4890 1048 1049


## 5. Train Neural Network

### 5.1 Define Neural Network Model

In [9]:
model = tf.keras.Sequential([
    Embedding(input_dim=10000, output_dim=64),
    Conv1D(filters=32, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.5),
    LSTM(64, kernel_regularizer=L1L2(0, 0.001)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

opt = tf.keras.optimizers.legacy.Adam(learning_rate=0.001)

model.summary()

Metal device set to: Apple M1 Pro

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 64)          640000    
                                                                 
 conv1d (Conv1D)             (None, None, 32)          6176      
                                                                 
 batch_normalization (BatchN  (None, None, 32)         128       
 ormalization)                                                   
                                                                 
 max_pooling1d (MaxPooling1D  (None, None, 32)         0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, None, 32)          0         
                                         

### 5.2 Compile Neural Network Model

In [10]:
model.compile(
    loss='binary_crossentropy', 
    optimizer=opt, 
    metrics=[
        'binary_accuracy'
    ]
)

### 5.3 Train Neural Network

In [11]:
earlystopper = EarlyStopping(patience=15, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.000001, verbose=1, cooldown=5)

history = model.fit(
    X_train, 
    y_train,
    validation_data=(X_val, y_val),
    epochs=300, 
    batch_size=100,
    verbose=1,
    shuffle=True,
    callbacks=[earlystopper, reduce_lr]
)

Epoch 1/300


2023-07-10 21:31:25.635436: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


49/49 [==============================] - 328s 7s/step - loss: 0.6103 - binary_accuracy: 0.7086 - val_loss: 0.6593 - val_binary_accuracy: 0.7912 - lr: 0.0010
Epoch 2/300
49/49 [==============================] - 333s 7s/step - loss: 0.3330 - binary_accuracy: 0.8851 - val_loss: 0.5531 - val_binary_accuracy: 0.7645 - lr: 0.0010
Epoch 3/300
49/49 [==============================] - 335s 7s/step - loss: 0.1870 - binary_accuracy: 0.9513 - val_loss: 0.3784 - val_binary_accuracy: 0.8942 - lr: 0.0010
Epoch 4/300
49/49 [==============================] - 333s 7s/step - loss: 0.1230 - binary_accuracy: 0.9744 - val_loss: 0.3472 - val_binary_accuracy: 0.8808 - lr: 0.0010
Epoch 5/300
49/49 [==============================] - 328s 7s/step - loss: 0.1410 - binary_accuracy: 0.9654 - val_loss: 1.5111 - val_binary_accuracy: 0.6006 - lr: 0.0010
Epoch 6/300
49/49 [==============================] - 332s 7s/step - loss: 0.0819 - binary_accuracy: 0.9861 - val_loss: 0.5835 - val_binary_accuracy: 0.8103 - lr: 0.001

KeyboardInterrupt: 

In [ ]:
fig = go.Figure(
    data = [
        go.Scatter(y=history.history['loss'], name="train"),
        go.Scatter(y=history.history['val_loss'], name="val"),
    ],
    layout = {"yaxis": {"title": "Loss [BCE]"}, "xaxis": {"title": "Epoch"}, "title": "Model Loss over Epochs"}
)
fig.show()


## 6. Validation of Results

In [ ]:
y_test_predictions = (np.array(model.predict(X_test)) >= 0.5).astype(int)
f1score = f1_score(y_test, y_test_predictions)
gm = geometric_mean_score(y_test, y_test_predictions, average="binary")
auc = roc_auc_score(y_test, y_test_predictions, average="weighted")
precision = precision_score(y_test, y_test_predictions)
recall = recall_score(y_test, y_test_predictions)

In [ ]:
data = [["F1-Score", "G-Mean", "AUC", "Precision", "Recall"], [f1score, gm, auc, precision, recall]] # Create list with values
table = PrettyTable(data[0]) # Generate table with metrics
table.add_rows(data[1:]) # Add data to table
print(table) # Show table